## Ch15. Foundation forecasting models Forecasting: Principles & Practice (Python Edition) Extracted from: fpppy-15-foundation-models.qmd

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## [Setup] Imports & Configuration --- Setup / Hidden in slides ---

In [2]:
import warnings; warnings.filterwarnings("ignore")
import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS
from utilsforecast.plotting import plot_series
from utilsforecast.evaluation import evaluate
from utilsforecast.losses import mae
plt.rcParams.update({"figure.figsize": (7, 3.5)})

2026-09-20 07:19:12,974	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-09-20 07:19:13,198	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


In [3]:
from nixtla import NixtlaClient


ModuleNotFoundError: No module named 'nixtla'

## [Slide 5] 15.2 NHITS Transfer Learning Example

In [4]:
from datasetsforecast.m4 import M4
from neuralforecast.utils import AirPassengersDF

Load M4 monthly data (source domain)

In [5]:
Y_df = M4.load(directory="data", group="Monthly")[0].assign(
    ds=lambda df: df.groupby("unique_id")["ds"].transform(
        lambda x: pd.date_range(
            start="1970-01-01", periods=len(x), freq="MS"
        )
    )
)
horizon = 12
stacks = 3
models = [NHITS(
    input_size=5 * horizon, h=horizon,
    max_steps=2_000,
    stack_types=stacks * ["identity"],
    n_blocks=stacks * [1],
    mlp_units=[[256, 256] for _ in range(stacks)],
    n_pool_kernel_size=stacks * [1],
    batch_size=32, scaler_type="standard",
    n_freq_downsample=[12, 4, 1],
)]
nf = NeuralForecast(models=models, freq="MS")
nf.fit(df=Y_df)

Seed set to 1


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


RuntimeError: The NVIDIA driver on your system is too old (found version 12000). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.

Zero-shot on unseen Air Passengers (target domain)

In [6]:
transfer_preds = nf.predict(df=AirPassengersDF.copy())

Exception: You must fit the model before predicting.

## [Slide 10] 15.3 Chronos (Amazon)

In [7]:
y = AirPassengersDF["y"].to_numpy()


In [8]:
from chronos import ChronosPipeline
import torch

pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)
forecast = pipeline.predict(
    context=torch.tensor(y).unsqueeze(0),
    prediction_length=12,
)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/amazon/chronos-t5-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/amazon/chronos-t5-small/a971ba21945c4f1796b17a91fe69214b5f4ad472/config.json?%2Famazon%2Fchronos-t5-small%2Fresolve%2Fmain%2Fconfig.json=&etag=%22b9ce8d0e4e0eee976236d60380470d74b210328b%22 "HTTP/1.1 200 OK"


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


INFO:httpx:HTTP Request: HEAD https://huggingface.co/amazon/chronos-t5-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/amazon/chronos-t5-small/a971ba21945c4f1796b17a91fe69214b5f4ad472/config.json?%2Famazon%2Fchronos-t5-small%2Fresolve%2Fmain%2Fconfig.json=&etag=%22b9ce8d0e4e0eee976236d60380470d74b210328b%22 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 2819.33it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/amazon/chronos-t5-small/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/amazon/chronos-t5-small/a971ba21945c4f1796b17a91fe69214b5f4ad472/generation_config.json?%2Famazon%2Fchronos-t5-small%2Fresolve%2Fmain%2Fgeneration_config.json=&etag=%227528dbb1b6ce860d242aff71294a5fef12a41572%22 "HTTP/1.1 200 OK"


TypeError: ChronosPipeline.predict() got an unexpected keyword argument 'context'

## [Slide 13] 15.4 Example: Electricity Price Forecasting

In [9]:
df = pd.read_csv(
    "data/electricity_short.csv", parse_dates=["ds"]
)

## [Slide 15] 15.4 TimeGPT Zero-Shot Forecasting

In [10]:
nixtla_client = NixtlaClient(api_key="YOUR_API_KEY")

NameError: name 'NixtlaClient' is not defined

Zero-shot: no fine-tuning, direct application

In [11]:
preds_df = nixtla_client.forecast(
    df=df, h=24, level=[80, 90]
)

NameError: name 'nixtla_client' is not defined

Cross-validation for honest evaluation

In [12]:
cv_preds_df = nixtla_client.cross_validation(
    df=df, h=24, n_windows=3
)

NameError: name 'nixtla_client' is not defined

Evaluate

In [13]:
evaluation = evaluate(
    cv_preds_df, models=["TimeGPT"], metrics=[mae]
)

NameError: name 'cv_preds_df' is not defined

## [Slide 17] 15.4 TimeGPT Fine-Tuning

In [14]:
cv_finetune_preds_df = nixtla_client.cross_validation(
    df=df,
    h=24,
    n_windows=3,
    finetune_steps=15,    # number of gradient steps
    finetune_loss="mae",  # target metric to optimise
)

NameError: name 'nixtla_client' is not defined

## [Slide 19] 15.4 TimeGPT with Exogenous Variables

In [15]:
future_ex_vars_df = pd.read_csv(
    "data/electricity_future_vars.csv",
    parse_dates=["ds"],
)

Forecast with future exogenous variables

In [16]:
exog_preds_df = nixtla_client.forecast(
    df=df,
    X_df=future_ex_vars_df,   # future covariates
    h=24,
    level=[80],
)

NameError: name 'nixtla_client' is not defined

## [Slide 21] 15.4 Moirai Zero-Shot Forecasting

In [17]:
from gluonts.dataset.pandas import PandasDataset
from gluonts.dataset.split import split
from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
import numpy as np

full_df = pd.concat([df, future_ex_vars_df], axis=0)
full_df = full_df.set_index("ds")
ds = PandasDataset.from_long_dataframe(
    full_df, target="y", item_id="unique_id",
    feat_dynamic_real=full_df.columns
        .drop(["unique_id", "y"]).tolist(),
)
train, test_template = split(ds, offset=-24)
test_data = test_template.generate_instances(
    prediction_length=24, windows=1, distance=24,
)
model = MoiraiForecast(
    module=MoiraiModule.from_pretrained(
        "Salesforce/moirai-1.0-R-large"
    ),
    prediction_length=24, context_length=240,
    patch_size="auto", num_samples=50,
    target_dim=1,
    feat_dynamic_real_dim=ds.num_feat_dynamic_real,
    past_feat_dynamic_real_dim=ds.num_past_feat_dynamic_real,
)
predictor = model.create_predictor(batch_size=32)
forecasts = list(predictor.predict(test_data.input))

ModuleNotFoundError: No module named 'gluonts'

## [Slide 23] 15.4 Moirai Predictions and Combining Forecasts

In [18]:
fc = forecasts[0]
moirai_preds_df = (
    pd.DataFrame(
        np.quantile(fc.samples, [0.5, 0.1, 0.9], axis=0).T
    )
    .set_axis(["Moirai", "Moirai-lo-80",
               "Moirai-hi-80"], axis="columns")
    .assign(ds=fc.index.to_timestamp())
)

NameError: name 'forecasts' is not defined

Combine TimeGPT and Moirai forecasts

In [19]:
fcst_df = preds_df.merge(exog_preds_df).merge(moirai_preds_df)

plot_series(
    df, fcst_df, max_insample_length=100,
    xlabel="Hour", ylabel="Price",
    title="Nord Pool electricity price",
    palette="black_and_3color", rm_legend=False,
    legend_loc="outside lower center",
)

NameError: name 'preds_df' is not defined

## [Slide 25] 15.4 Chronos Usage

In [20]:
from chronos import ChronosPipeline
import torch

Load pre-trained model (T5-small variant)

In [21]:
pipeline = ChronosPipeline.from_pretrained(
    "amazon/chronos-t5-small",
    device_map="cpu",
    torch_dtype=torch.bfloat16,
)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/amazon/chronos-t5-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/amazon/chronos-t5-small/a971ba21945c4f1796b17a91fe69214b5f4ad472/config.json?%2Famazon%2Fchronos-t5-small%2Fresolve%2Fmain%2Fconfig.json=&etag=%22b9ce8d0e4e0eee976236d60380470d74b210328b%22 "HTTP/1.1 200 OK"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/amazon/chronos-t5-small/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/amazon/chronos-t5-small/a971ba21945c4f1796b17a91fe69214b5f4ad472/config.json?%2Famazon%2Fchronos-t5-small%2Fresolve%2Fmain%2Fconfig.json=&etag=%22b9ce8d0e4e0eee976236d60380470d74b210328b%22 "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 1774.95it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/amazon/chronos-t5-small/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"


INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/amazon/chronos-t5-small/a971ba21945c4f1796b17a91fe69214b5f4ad472/generation_config.json?%2Famazon%2Fchronos-t5-small%2Fresolve%2Fmain%2Fgeneration_config.json=&etag=%227528dbb1b6ce860d242aff71294a5fef12a41572%22 "HTTP/1.1 200 OK"


y: numpy array or list of historical values

In [22]:
y = AirPassengersDF["y"].to_numpy()


In [23]:
forecast = pipeline.predict(
    context=torch.tensor(y).unsqueeze(0),
    prediction_length=12,
    num_samples=20,    # number of sample paths
)

TypeError: ChronosPipeline.predict() got an unexpected keyword argument 'context'

forecast shape: [num_series, num_samples, prediction_length]

In [24]:
low, median, high = np.quantile(
    forecast[0].numpy(), [0.1, 0.5, 0.9], axis=0
)

NameError: name 'forecast' is not defined